# Module C : The Curse of Dimensionality & The Ansatz

In Module 2, you solved the Schrödinger equation exactly by building a grid (finite difference). It felt powerful, right? 

Unfortunately, we have to abandon that method immediately. It is physically impossible to use grid methods for real chemistry. Here is why.

## The Scaling Problem

Suppose we want to simulate a single electron in a 3D box. If we use 100 grid points along the X, Y, and Z axes, our total grid has $100 \times 100 \times 100 = 1,000,000$ points. Our Hamiltonian matrix would be $10^6 \times 10^6$. That requires about 8 Terabytes of RAM just to store in memory.

Now, what if we want to simulate the two electrons in a Helium atom? The wavefunction depends on the coordinates of *both* electrons simultaneously: $\psi(x_1, y_1, z_1, x_2, y_2, z_2)$. 
Our grid now requires $100^6 = 1,000,000,000,000$ points. 

This exponential blow-up is called the **Curse of Dimensionality**. Exact numerical grids are dead on arrival for anything larger than a Hydrogen atom.

## Simplifying the Physics: The Born-Oppenheimer Approximation

To survive, computational chemists have to make approximations. The first major approximation targets the **Hamiltonian operator** ($\hat{H}$). 

A true molecule consists of moving electrons and moving nuclei. The total Kinetic Energy ($\hat{T}$) would include the movement of everything. But protons and neutrons are thousands of times heavier than electrons. From the perspective of a zipping electron, the massive nuclei appear to be completely frozen in place.

This is the **Born-Oppenheimer (BO) Approximation**. We assume the nuclear kinetic energy is zero ($\hat{T}_{Nuc} = 0$). We "freeze" the nuclei at specific 3D coordinates, calculate the energy of the electrons zipping around that static framework, and then slowly move the nuclei to new positions (Geometry Optimization!).

## Simplifying the Math: The Variational Theorem

Even with frozen nuclei, we still cannot exactly solve $\hat{H}\psi = E\psi$ for multiple electrons. So, what do we do? We **guess**.

The Variational Theorem is the most important mathematical safety net in quantum chemistry. It states that if you guess *any* valid wavefunction (called a trial wavefunction or **Ansatz**), the energy of your guess will ALWAYS be greater than or equal to the true ground state energy:

$$ E_{\text{guess}} \ge E_{\text{true}} $$

But how do we calculate the energy of a guess? Because our guess isn't the true eigenfunction, we can't just pull $E$ out of $\hat{H}\psi = E\psi$. Instead, we calculate the **Expectation Value** (the quantum mechanical average) by sandwiching the Hamiltonian between our guessed wavefunction and integrating over all space:

$$ E_{\text{guess}} = \frac{\int \psi^* \hat{H} \psi \, d\tau}{\int \psi^* \psi \, d\tau} = \frac{\langle \psi | \hat{H} | \psi \rangle}{\langle \psi | \psi \rangle} $$

Notice the denominator! In Week 2, you spent a lot of time manually normalizing your wavefunctions so that $\int \psi^* \psi \, d\tau = 1$. By including the overlap integral $\langle \psi | \psi \rangle$ in the denominator of our energy equation, the math automatically scales the answer. **You never have to manually normalize a Variational guess!**



The Variational Theorem (we will not prove this theorem in this course) is the most important mathematical safety net in quantum chemistry. It states that if you guess *any* valid wavefunction (called a trial wavefunction or **Ansatz**), the energy of your guess will ALWAYS be greater than or equal to the true ground state energy:

$$ E_{\text{guess}} \ge E_{\text{true}} $$

This gives us a brilliant strategy:
1. Guess a mathematical function with a tunable parameter (like the width of the wave).
2. Calculate the energy.
3. Use an optimization algorithm (like we learned in Chapter 1) to tweak the parameter until the energy hits a minimum.

## Let Python Do the Calculus

Looking at that equation above, you might be panicking. It requires taking the second derivative of a complex guess (to apply the Kinetic Energy operator inside $\hat{H}$) and then doing a massive integral from $-\infty$ to $\infty$.

You will *not* be looking up integral tables in this course. You already learned how to use `SymPy` in Week 1. You can hand your guessed wavefunction and your Hamiltonian directly to the computer, and it will do the analytical calculus for you. 

Look at how easily SymPy handles the normalization denominator for a Gaussian guess $\psi(x) = e^{-\alpha x^2}$:

In [ ]:
import sympy as sp

# Define our symbolic variables. 
# We tell SymPy they are positive and real so it can simplify the math!
x, alpha = sp.symbols('x alpha', real=True, positive=True)

# Define our guess
psi = sp.exp(-alpha * x**2)

# Calculate the denominator: integral of (psi * psi) from -infinity to infinity
overlap_integral = sp.integrate(psi * psi, (x, -sp.oo, sp.oo))

print("The analytical overlap integral is:")
display(overlap_integral)

##  Building the Ansatz: Slater vs. Gaussian

We don't guess blindly. A valid wavefunction must follow physical rules: it must decay to zero as it moves infinitely far from the nucleus, and it must be smooth so its kinetic energy doesn't explode.

Because we know the exact analytical solution for the Hydrogen atom, we know exactly what an electron orbital *should* look like. The exact spatial ground state of Hydrogen is a **Slater-type orbital**, which decays exponentially:
$$ \psi_{\text{Slater}}(r) \propto e^{-r} $$

However, Slater orbitals have a major computational flaw: calculating the energy integrals for two Slater orbitals on different atoms is mathematically excruciating for a computer. 

To speed up the computer, computational chemists made a compromise. Instead of the exact $e^{-r}$, we guess a **Gaussian function**:
$$ \psi_{\text{Gaussian}}(r) \propto e^{-\alpha r^2} $$
Here, $\alpha$ is our tunable parameter. Gaussian integrals are magically easy for computers to solve analytically. But this speed comes at a physical cost. Let's visualize our guess compared to the true analytical answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Distance from the nucleus (r = 0 is the nucleus)
r = np.linspace(0, 5, 200)

# The Exact Analytical Hydrogen Wavefunction (Slater)
psi_slater = np.exp(-r)

# Our Trial Wavefunction (Gaussian) with a tuned alpha parameter
alpha = 0.28
psi_gaussian = 0.7 * np.exp(-alpha * r**2) # Scaled slightly for visual comparison

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(r, psi_slater, color='black', linewidth=2, label='True Answer (Slater: $e^{-r}$)')
plt.plot(r, psi_gaussian, color='crimson', linewidth=2, linestyle='--', label=r'Our Guess (Gaussian: $e^{-\alpha r^2}$)')

# Highlight the nucleus
plt.axvline(0, color='gray', linestyle=':', alpha=0.5)

plt.title('Comparing the True Wavefunction to a Gaussian Guess')
plt.xlabel('Distance from Nucleus $r$ (Bohr)')
plt.ylabel('Wavefunction Amplitude $\psi(r)$')
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define our grid just for plotting
x = np.linspace(-5, 5, 200)

# Define our Gaussian guess function
def trial_wavefunction(x, alpha):
    return np.exp(-alpha * x**2)

# Plot the guess with different values of alpha
plt.figure(figsize=(7, 4))
plt.plot(x, trial_wavefunction(x, 0.2), label=r'$\alpha = 0.2$ (Wide)')
plt.plot(x, trial_wavefunction(x, 1.0), label=r'$\alpha = 1.0$ (Medium)')
plt.plot(x, trial_wavefunction(x, 3.0), label=r'$\alpha = 3.0$ (Narrow)')

plt.title('Our Trial Wavefunctions (Gaussians)')
plt.xlabel('Position (x)')
plt.ylabel(r'$\psi(x)$')
plt.legend()
plt.show()

### The Cusp Condition
Look closely at $r=0$ (the nucleus) in the plot above. 
The exact analytical Slater orbital has a sharp point, or a **cusp**. The electron density spikes dramatically right at the nucleus. 
Our Gaussian guess, however, has a flat top. It completely misses this nuclear cusp! Furthermore, the Gaussian drops off too quickly (the tail is too thin) as $r$ increases.

If a single Gaussian is a somewhat "bad" physical guess, why do we use it? Because we can mathematically stack them. In the next module, you will learn how to use a **Linear Combination**—adding a narrow Gaussian (to fake the cusp) and a wide Gaussian (to fake the tail) together. 

But before we sum multiple Gaussians, we need to prove we can optimize just one!

## 6. The Game Plan

In the upcoming problem set, you will apply this entire workflow to the 1D Harmonic Oscillator. 

You already solved the Harmonic Oscillator exactly in Module 2 using a massive matrix grid. Now, you will solve it the Variational way:
1.  You will write a function that accepts a Gaussian parameter $\alpha$.
2.  You will use SymPy to analytically calculate the Expectation Value $\frac{\langle \psi | \hat{H} | \psi \rangle}{\langle \psi | \psi \rangle}$ of that Gaussian guess.
3.  You will hand that exact energy equation to `scipy.optimize.minimize`.
4.  The algorithm will walk downhill until it finds the optimal $\alpha$ that yields the lowest possible energy. 

You are about to write your first true variational quantum mechanics solver!